### **1. Analysising Data With Pandas** 

In [184]:
import pandas as pd

#### **A. Load the clean Dataset**

In [185]:
df= pd.read_csv('dataset/kakuma_fresh_mart_sales_clean.csv',parse_dates=['date'])

In [186]:
df.head(10)

,receipt,date,branch,product,category,qty,price,payment,customer,is_return,unusual_qty,new_customer
0,R10001,2026-10-01,Kakuma Town,Water,drinks,2.0,50.0,M-Pesa,NaN,False,False,True
1,R10002,2026-10-01,Lodwar,Water,drinks,4.0,50.0,M-Pesa,NaN,False,False,True
2,R10003,2026-10-01,Kakuma Town,Salt,cooking,2.0,40.0,Cash,NaN,False,False,True
3,R10004,2026-10-01,Kitale,Milk,dairy,2.0,65.0,Cash,C0308,False,False,False
4,R10005,2026-10-01,Kakuma Town,Detergent,household,3.0,270.0,Cash,NaN,False,False,True
5,R10006,2026-10-01,Lodwar,Water,drinks,2.0,50.0,Cash,NaN,False,False,True
6,R10007,2026-10-01,Lodwar,Cooking oil,cooking,4.0,320.0,M-Pesa,NaN,False,False,True
7,R10008,2026-10-01,Lodwar,Soda,drinks,2.0,70.0,M-Pesa,C0067,False,False,False
8,R10009,2026-10-01,Lodwar,Wheat flour,grains,3.0,190.0,M-Pesa,C0122,False,False,False
9,R10010,2026-10-01,Kakuma Town,Soap,household,4.0,85.0,Cash,C0181,False,False,False


In [187]:
# create Revenue column
df['revenue']=df['qty'] * df['price']

In [188]:
df.head()

,receipt,date,branch,product,category,qty,price,payment,customer,is_return,unusual_qty,new_customer,revenue
0,R10001,2026-10-01,Kakuma Town,Water,drinks,2.0,50.0,M-Pesa,NaN,False,False,True,100.0
1,R10002,2026-10-01,Lodwar,Water,drinks,4.0,50.0,M-Pesa,NaN,False,False,True,200.0
2,R10003,2026-10-01,Kakuma Town,Salt,cooking,2.0,40.0,Cash,NaN,False,False,True,80.0
3,R10004,2026-10-01,Kitale,Milk,dairy,2.0,65.0,Cash,C0308,False,False,False,130.0
4,R10005,2026-10-01,Kakuma Town,Detergent,household,3.0,270.0,Cash,NaN,False,False,True,810.0


In [189]:
df.columns

Index(['receipt', 'date', 'branch', 'product', 'category', 'qty', 'price',
       'payment', 'customer', 'is_return', 'unusual_qty', 'new_customer',
       'revenue'],
      dtype='str')

In [190]:
# remove unusual Quantity(999,500,250)
sales= df[df['unusual_qty']==False]

In [191]:
# net Revenue
print('Net Revenue:',sales['revenue'].sum())

Net Revenue: 929664.0


#### **2.where does the revenue comes from??**
> **question 1**:which branch brings in most revenue and is it because customers spend more or because there are more of them

In [192]:
# revenue by branches
sales.groupby('branch')['revenue'].sum().sort_values(ascending=False)

branch
Kakuma Town    365025.0
Kitale         283236.0
Lodwar         281403.0
Name: revenue, dtype: float64

'kakuma town has the highest revenue (365025.0)'-40% compare to lodwar (281403.0)
and kitale (281403.0)

In [193]:
# branch summary_ Revenue,Orders,AOV
branch_summary= sales.groupby('branch').agg(
    total_revenue= ('revenue','sum'),
    Customers=('receipt','count'),
    AOV=('revenue','mean')             
)
branch_summary.sort_values('total_revenue',ascending=False).round().reset_index

<bound method DataFrame.reset_index of              total_revenue  Customers    AOV
branch                                      
Kakuma Town       365025.0       1000  374.0
Kitale            283236.0        737  389.0
Lodwar            281403.0        727  396.0>


**Kakuma Town earns about 40% of revenue (KES 375,795 of 949,214), but because it
sells more, not because customers spend more. It has 1,000 sales lines against about
730 elsewhere, and its average per line (KES 376) is the lowest of the three branches.

3: What Drives Revenue?

Question 2: Which categories and products drive revenue? Here the useful measure is not
the amount

In [194]:
# revenue by cataegory
cat_revenue= sales.groupby('category')['revenue'].sum().sort_values(ascending=False)

# get the percentage
cat_revenue_pct= (cat_revenue/cat_revenue.sum()*100).round(1)

# top 5 category
cat_revenue_pct.head(5)

category
grains       25.2
cooking      24.3
dairy        22.5
household    11.0
drinks       10.3
Name: revenue, dtype: float64

three categories Grains,cookings and dairys,account for the about 72% of the revenues (25.2,24.3 and 22.5)
    produces the smallest accounts for only 6.5 percent.

In [195]:
# revenue by products
product_revenue=sales.groupby('product')['revenue'].sum().sort_values(ascending=False)

product_revenue_pct= product_revenue/product_revenue.sum()*100

#top 5 product
product_revenue_pct.round(2).head(5)


product
Eggs           14.81
Cooking oil    13.36
Rice            9.73
Wheat flour     8.63
Detergent       7.80
Name: revenue, dtype: float64

 Revenue Vs Volume

In [196]:
# orders by product
product_orders= sales.groupby('product')['qty'].sum().sort_values(ascending=False)

# top 5 products
product_orders.head(10)

product
Water          552.0
Soda           505.0
Milk           500.0
Wheat flour    425.0
Sugar          413.0
Cooking oil    401.0
Maize flour    362.0
Rice           328.0
Soap           318.0
Eggs           298.0
Name: qty, dtype: float64

**Five products out of nineteen bring in 54 percent of the revenue. This is concentration: a
few items matter a lot. The manager should make sure those five never run out of stock.

### **4.Do customers behave Differently?**

Question 3: Do returning customers spend differently from others from other shoppers,and how do customers pay?

In [197]:
# new vs returning customers
sales['new_customer'].value_counts()

new_customer
True     1367
False    1097
Name: count, dtype: int64

In [198]:
# customer summary
customer_summary= sales.groupby('new_customer')['revenue'].agg(['count','mean','median'])

customer_summary.round(2).reset_index()


,new_customer,count,mean,median
0,False,1070,383.53,250.0
1,True,1344,386.38,260.0


returning average KES 386 per line and new customers KES 384. The medians are 250 and
260. These are practically the same. There is no sign that loyalty members spend more
per line. That is a real finding, and an important one: "no difference" is a result, not a
failure.

### **payment behaviours**

In [199]:
#payment methods in each branches

branch_payment=pd.crosstab(sales['branch'],sales['payment'],normalize='index')*100
branch_payment.round(1)

payment,Card,Cash,Credit,M-Pesa,unknown
branch,,,,,
Kakuma Town,5.9,45.6,3.6,43.0,1.9
Kitale,7.7,46.3,2.6,40.3,3.1
Lodwar,7.3,41.7,3.6,44.8,2.6


In [200]:
sales.pivot_table(
    index='branch',
    columns=['payment'],
    aggfunc='size',
    
)

payment,Card,Cash,Credit,M-Pesa,unknown
branch,,,,,
Kakuma Town,59,456,36,430,19
Kitale,57,341,19,297,23
Lodwar,53,303,26,326,19


### **5.When Do we Sell most**

In [201]:
# daily revenue
daily= sales.groupby('date')['revenue'].sum().reset_index()

# get days names
daily['weekday']= daily['date'].dt.day_name()
daily.head(10)

,date,revenue,weekday
0,2026-10-01,37292.0,Thursday
1,2026-10-02,41719.0,Friday
2,2026-10-03,53495.0,Saturday
3,2026-10-04,50242.0,Sunday
4,2026-10-05,42711.0,Monday
5,2026-10-06,46697.0,Tuesday
6,2026-10-07,43872.0,Wednesday
7,2026-10-08,36524.0,Thursday
8,2026-10-09,35151.0,Friday
9,2026-10-10,60891.0,Saturday


In [202]:
# daily salary
days_orders= ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
daily_sales= daily.groupby('weekday')['revenue'].mean().reindex(days_orders)

daily_sales.reset_index().round(2)

,weekday,revenue
0,Monday,42659.33
1,Tuesday,43159.00
2,Wednesday,39350.33
3,Thursday,35358.67
4,Friday,37474.33
5,Saturday,54855.67
6,Sunday,57030.67


In [203]:
#compare weekends vs week days average revenue
daily['is_weekend']= daily['weekday'].isin(['saturday','sunday'])

daily.groupby('is_weekend')['revenue'].mean().reset_index().round(2)

,is_weekend,revenue
0,False,44269.71


Do weekends perform better in all branches

In [204]:
sales['is_weekend']= sales['date'].dt.day_of_week >=5
pvt=pd.pivot_table(
    sales,
    index='branch',
    columns='is_weekend',
    values='revenue',
    aggfunc='sum'
)
pvt.round(1)

is_weekend,False,True
branch,,
Kakuma Town,235484.0,129541.0
Kitale,180658.0,102578.0
Lodwar,177863.0,103540.0


In [205]:
days_per_group=sales.groupby('is_weekend')['date'].nunique()
per_day= pvt/days_per_group
per_day.round(0)

is_weekend,False,True
branch,,
Kakuma Town,15699.0,21590.0
Kitale,12044.0,17096.0
Lodwar,11858.0,17257.0






Insight 4: Timing
Weekend days earn about 41% more than weekdays: KES 57,141 a day against
40,425. The lift is present in every branch (38% at Kakuma Town, 42% at Kitale, 45% at
Lodwar). So what: schedule more staff and make sure stock is full for Saturday and
Sunday. Caveat: the data covers only three weeks, with each weekday seen three
times, so differences between individual weekdays are weak evidence.

## Findings summary
**Bottom line:** Sales are not weak at any one branch. The business is concentrated
in a few products and in the weekend, so staffing and stock should follow those.
1. Kakuma Town earns about 40% of revenue because of volume, not higher spending.
2. Five of 19 products bring in 54% of revenue. Eggs leads by revenue, not by units.
3. Loyalty members do not spend more per line, and payments are alike everywhere.
4. Weekend days earn about 41% more than weekdays, in every branch.
**Please confirm:** the three extreme quantities, and day-first slash dates.
**Not covered:** customers and visits, because the file records items only.